#### Import numpy & keras

In [16]:
import numpy as np
import keras
from datasets import load_dataset, DatasetDict, Image, Dataset
import datetime

In [17]:
import tensorflow as tf
print(tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))

2.21.0
Num GPUs Available:  0
[]


### Bilder normalisieren

In [18]:
def transform(example):
    image = np.array(example["image"], dtype=np.float32) / 255.0
    return {"image": image, "label": example["label"]}


#### 1. get training data

In [19]:
import matplotlib.pyplot as plt
import PIL
print(PIL.__version__)

ds = load_dataset("jonathan-roberts1/NWPU-RESISC45")
print(ds.shape)

train_data = ds["train"]
split_1 = train_data.train_test_split(
    test_size=0.15,
    seed=42,          # sorgt dafür, dass Validation immer gleich bleibt
    shuffle=True
)

validation_dataset = split_1["test"]

12.2.0
{'train': (31500, 2)}


In [20]:
remaining_dataset = split_1["train"]
split_2 = remaining_dataset.train_test_split(
    test_size=0.25,
    seed=42,
    shuffle=True
)

train_ds = split_2["train"]

In [21]:
from datasets import concatenate_datasets
import random

# -----------------------------
# AUGMENTER (keras statt tf.keras)
# -----------------------------
augmenter = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.5),
    keras.layers.RandomZoom(height_factor=(-0.1, 0.1), width_factor=(-0.1, 0.1)),
    keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    keras.layers.RandomContrast(0.15),
])

# -----------------------------
# AUGMENTATIONSANTEIL
# -----------------------------
augment_fraction = 0.2
num_augmented = int(len(train_ds) * augment_fraction)

indices = random.sample(range(len(train_ds)), num_augmented)
subset_to_augment = train_ds.select(indices)

# -----------------------------
# BATCH AUGMENTATION FUNKTION
# -----------------------------
def augment_batch(batch):
    images = np.array(batch["image"], dtype=np.float32)

    images = keras.ops.convert_to_tensor(images)
    images = augmenter(images, training=True)

    images = keras.ops.clip(images, 0, 255)
    images = keras.ops.convert_to_numpy(images).astype(np.uint8)

    return {
        "image": images,
        "label": np.array(batch["label"], dtype=np.int32)
    }

# -----------------------------
# MAP (batch processing)
# -----------------------------
augmented_ds = subset_to_augment.map(
    augment_batch,
    batched=True,
    batch_size=32
)

# -----------------------------
# CONCATENATE
# -----------------------------
train_ds = concatenate_datasets([train_ds, augmented_ds])

# -----------------------------
# OUTPUT
# -----------------------------
print("Originale Trainingsdaten:", len(split_2["train"]))
print("Nach Augmentierung:", len(train_ds))

Map:   0%|          | 0/4016 [00:00<?, ? examples/s]

Originale Trainingsdaten: 20081
Nach Augmentierung: 24097


In [22]:
from datasets import DatasetDict

test_ds = split_2["test"]

# DatasetDict erzeugen
final_dataset = DatasetDict({
    "train": train_ds,
    "validation": validation_dataset,
    "test": test_ds
})

# Optional nur falls image-feature kaputt ist
# from datasets import Image
# final_dataset = final_dataset.cast_column("image", Image())

# Transformation anwenden
final_dataset = final_dataset.with_transform(transform)

print(final_dataset)

# TensorFlow Datasets
tf_train = final_dataset["train"].to_tf_dataset(
    columns="image",
    label_cols="label",
    batch_size=128,
    shuffle=True
)

tf_test = final_dataset["test"].to_tf_dataset(
    columns="image",
    label_cols="label",
    batch_size=128
)

# Klassen
class_names = final_dataset["train"].features["label"].names
print(class_names)

# Bildform prüfen
img_shape = np.array(final_dataset["train"][0]["image"]).shape
print(img_shape)

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 24097
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 4725
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 6694
    })
})
['airplane', 'airport', 'baseball diamond', 'basketball court', 'beach', 'bridge', 'chaparral', 'church', 'circular farmland', 'cloud', 'commercial area', 'dense residential', 'desert', 'forest', 'freeway', 'golf course', 'ground track field', 'harbor', 'industrial area', 'intersection', 'island', 'lake', 'meadow', 'medium residential', 'mobile home park', 'mountain', 'overpass', 'palace', 'parking lot', 'railway', 'railway station', 'rectangular farmland', 'river', 'roundabout', 'runway', 'sea ice', 'ship', 'snowberg', 'sparse residential', 'stadium', 'storage tank', 'tennis court', 'terrace', 'thermal power station', 'wetland']
(256, 256, 3)


#### 2. define architecture

In [23]:
# Import a model from /models/*
## Todo: adjust modelname
from models.leNet_5 import generateModel
model_name = "leNet_5"

load model

In [24]:
model = generateModel(img_shape)
#model = keras.models.load_model("./models/" + model_name + ".keras")
model.summary()

C:\Users\marie\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 252, 252, 6)    │           456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 126, 126, 6)    │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 122, 122, 16)   │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 61, 61, 16)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 57, 57, 32)     │        12,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_2             │ (None, 28, 28, 32)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 32)     │        25,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_3             │ (None, 12, 12, 32)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 32)       │        25,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_4             │ (None, 4, 4, 32)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 45)             │         5,805 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 401,093 (1.53 MB)

 Trainable params: 401,093 (1.53 MB)

 Non-trainable params: 0 (0.00 B)

#### 3. set training parameter and fit model

non-specific callbacks

In [25]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=2,
    verbose=1,
    min_lr=1e-6
)

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

log_dir = "logs/" + model_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

savepath = "./models/" + model_name + ".keras"
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=savepath,
    save_freq="epoch",
    save_best_only=False,
    verbose=1
)

model.fit(
    tf_train,
    validation_data=tf_test,
    epochs=40,
    callbacks=[tensorboard_callback, early_stopping, reduce_lr],
)

model.save(savepath)

Epoch 1/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 57s 293ms/step - accuracy: 0.1593 - loss: 3.1287 - val_accuracy: 0.2139 - val_loss: 2.8806 - learning_rate: 0.0010
Epoch 2/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 56s 294ms/step - accuracy: 0.2325 - loss: 2.8156 - val_accuracy: 0.1839 - val_loss: 3.0358 - learning_rate: 0.0010
Epoch 3/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 55s 289ms/step - accuracy: 0.2600 - loss: 2.6904 - val_accuracy: 0.2599 - val_loss: 2.7507 - learning_rate: 0.0010
Epoch 4/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 54s 284ms/step - accuracy: 0.2906 - loss: 2.5648 - val_accuracy: 0.2737 - val_loss: 2.6388 - learning_rate: 0.0010
Epoch 5/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 53s 282ms/step - accuracy: 0.3168 - loss: 2.4632 - val_accuracy: 0.3177 - val_loss: 2.4820 - learning_rate: 0.0010
Epoch 6/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 55s 291ms/step - accuracy: 0.3524 - loss: 2.3206 - val_accuracy: 0.3219 - val_loss: 2.4505 - learning_rate: 0.0010
Epoch 7/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 55s 293ms/step - accuracy: 0.3

# Model 2 - four-block-cnn

In [ ]:
## Todo: adjust modelname
from models.four_block_cnn import generateModel
model_name = "four_block_cnn"
model = generateModel(img_shape)
#model = keras.models.load_model("./models/" + model_name + ".keras")
model.summary()
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

log_dir = "logs/" + model_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

savepath = "./models/" + model_name + ".keras"
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=savepath,
    save_freq="epoch",
    save_best_only=False,
    verbose=1
)

model.fit(
    tf_train,
    validation_data=tf_test,
    epochs=15,
    callbacks=[tensorboard_callback, early_stopping, reduce_lr],
)

model.save(savepath)

# Model 3 - five_block_v1

In [ ]:
## Todo: adjust modelname
from models.five_block_v1 import generateModel
model_name = "five_block_v1"
model = generateModel(img_shape)
#model = keras.models.load_model("./models/" + model_name + ".keras")
model.summary()
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

log_dir = "logs/" + model_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

savepath = "./models/" + model_name + ".keras"
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=savepath,
    save_freq="epoch",
    save_best_only=False,
    verbose=1
)

model.fit(
    tf_train,
    validation_data=tf_test,
    epochs=20,
    callbacks=[tensorboard_callback, early_stopping, reduce_lr],
)

model.save(savepath)

# Model 4 - five_block_v2

In [ ]:
## Todo: adjust modelname
from models.five_block_v2 import generateModel
model_name = "five_block_v2"
model = generateModel(img_shape)
#model = keras.models.load_model("./models/" + model_name + ".keras")
model.summary()
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

log_dir = "logs/" + model_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

savepath = "./models/" + model_name + ".keras"
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=savepath,
    save_freq="epoch",
    save_best_only=False,
    verbose=1
)

model.fit(
    tf_train,
    validation_data=tf_test,
    epochs=20,
    callbacks=[tensorboard_callback, early_stopping, reduce_lr],
)

model.save(savepath)

# Model 5 - resNet

In [ ]:
## Todo: adjust modelname
from models.resNet import generateModel
model_name = "resNet"
model = generateModel(img_shape)
#model = keras.models.load_model("./models/" + model_name + ".keras")
model.summary()
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

log_dir = "logs/" + model_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

savepath = "./models/" + model_name + ".keras"
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=savepath,
    save_freq="epoch",
    save_best_only=False,
    verbose=1
)

model.fit(
    tf_train,
    validation_data=tf_test,
    epochs=5,
    callbacks=[tensorboard_callback, early_stopping, reduce_lr],
)

model.save(savepath)

In [ ]:
%load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir logs

#### 4. predict output 

In [ ]:
import random

# Zufälligen Index wählen
idx = random.randint(0, len(final_dataset["test"]) - 1)

# Sample holen
sample = final_dataset["test"][idx]

# Bild und echtes Label
img = sample["image"]
true_idx = sample["label"]

# Batch-Dimension hinzufügen
input_img = np.expand_dims(img, axis=0)

# Prediction
prediction = model.predict(input_img, verbose=0)

# Vorhersageklasse
predicted_idx = np.argmax(prediction)

# Ausgabe
print("Index:", idx)
print("Predicted:", class_names[predicted_idx])
print("True:", class_names[true_idx])

# Bild anzeigen
plt.imshow(img)
plt.title(f"Pred: {class_names[predicted_idx]} | True: {class_names[true_idx]}")
plt.axis("off")
plt.show()